# Stacking Regressor on California Housing Dataset:

This notebook implements an ensemble **Stacking Regressor** using `scikit-learn` to predict median house values from the California Housing dataset.

### Architecture Overview:
* **Base Learners (Level-0):**
  * `LinearRegression` (wrapped with `StandardScaler` in a pipeline)
  * `DecisionTreeRegressor` (`max_depth=8`)
  * `GradientBoostingRegressor`
* **Meta-Learner (Level-1):** `KNeighborsRegressor` trained on 10-fold out-of-fold predictions (`cv=10`).

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor

from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error

## 1. Load & Split the Dataset:
Load the California Housing dataset and split it into **80% training** and **20% testing** sets.

In [2]:
X, y = fetch_california_housing(return_X_y=True)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Define Diverse Base Estimators:
Combine linear, single-tree, and gradient-boosted models so the ensemble captures both global linear trends and non-linear feature interactions. Feature scaling is applied via `make_pipeline` specifically for `LinearRegression`.

In [4]:
estimators = [
    ('lr', make_pipeline(StandardScaler(), LinearRegression())),
    ('dtreg', DecisionTreeRegressor(max_depth=8, random_state=42)),
    ('gdboost', GradientBoostingRegressor(random_state=42))
]

## 3. Train the Stacking Regressor:
Fit `StackingRegressor` using 10-fold cross-validation (`cv=10`) to generate out-of-fold predictions from the base estimators, which serve as input features for the `KNeighborsRegressor` meta-learner.

In [ ]:
st_reg = StackingRegressor(
    estimators = estimators,
    final_estimator = KNeighborsRegressor(),
    cv = 10
    )

st_reg.fit(X_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('lr', ...), ('dtreg', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",KNeighborsRegressor()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[Pipeline(step...egression())]), DecisionTreeR...ndom_state=42), GradientBoost...ndom_state=42)]"
final_estimator_ final_estimator_: estimatorThe regressor fit on the output of `estimators_` and responsible forfinal predictions.,KNeighborsRegressor,KNeighborsRegressor()
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,8
named_estimators_ named_estimators_: :class:`~sklearn.utils.Bunch`Attribute to access any fitted sub-estimators by name.,Bunch,{'lr': Pipeli...dom_state=42)}


## 4. Evaluate Model Performance:
Generate predictions on the unseen test set and evaluate accuracy using **$R^2$ Score** and **Root Mean Squared Error (RMSE)**.

In [ ]:
y_pred = st_reg.predict(X_test)

print("R2 Score:", r2_score(y_test, y_pred))
print("RMSE:", root_mean_squared_error(y_test, y_pred))

R2 Score: 0.7426559890627078
RMSE: 0.5807115483859021
